In [ ]:
# Databricks notebook sourceMAGIC %sqlselect * from bupa_bronze.providers

%md# Goal: Standardize provider keys and fraud flag; dedupe; (optional) normalize region/type if you have those columns.## Steps### 1.Read Bronze### 2.Type casting### 3.Key check### 4.DeduplicateWrite + register

In [ ]:
MAGIC %run /Workspace/Repos/karthikvanapalli1505@gmail.com/Bupa_Insuarnce_Project/02_Silver_Layer/_00_utils_silver

In [ ]:
# ============ PROVIDERS → SILVER ============prov_bz = spark.read.format("delta").load(paths_bronze["providers"])# Minimal schema used in Claims FK: Provider_ID, Fraud_Flagprov = (prov_bz  .select(F.col("Provider_ID").cast("string"),          F.col("Fraud_Flag").cast("int"))  .filter(F.col("Provider_ID").isNotNull()))# Dedupeprov_silver = prov.dropDuplicates(["Provider_ID"])(prov_silver.write.format("delta").mode("overwrite").save(paths_silver["providers"]))#spark.sql(f"CREATE TABLE IF NOT EXISTS {CATALOG_SILVER}.providers USING DELTA LOCATION '{paths_silver['providers']}'")# Save DataFrames as tables in the databaseprov_silver.write.format("delta").mode("overwrite").saveAsTable(f"{CATALOG_SILVER}.providers")print("Providers → Silver complete.")

In [ ]:
providers_df = spark.sql(f"SELECT * FROM {CATALOG_SILVER}.providers")display(providers_df)